# 04 D_content 子集专属评测实验

**目的**：在仅 D_content 文档（1000 docs，943 有内容视图）上重新评测所有方法，消除全局稀释效应，直接对比元数据 vs 内容 vs 融合的真实效果。

**背景**：NB03 在全局 521K 文档上评测，内容视图仅覆盖 0.18%，导致融合方法无法超越 Meta-only 基线。D_content 子集有 100% 三银标准覆盖率（Tag/Desc/Creator 均为 1000/1000），是理想的对照实验环境。

**评测方法**：
1. Meta-only（Fused3）
2. Content-only（S_tabcontent）
3. Naive Fusion
4. Adaptive Fusion
5. Adaptive+Consistency
6. Tag-only
7. Text-only
8. Behavior-only

**输出**：`tmp/content/subset_eval/`

## 1. 配置与导入

In [1]:
import os, sys, json, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from scipy import sparse

ROOT = Path(".").resolve().parent.parent
sys.path.insert(0, str(ROOT))

from src.metrics import (
    dcg_at_k, ndcg_at_k, average_precision_at_k, mrr_at_k,
    precision_at_k, recall_at_k,
)
from src.content.similarity import (
    load_manifest_flexible, load_csr_from_manifest,
)

warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# 路径
TMP_DIR     = ROOT / "tmp"
CONTENT_DIR = TMP_DIR / "content"
OUT_DIR     = CONTENT_DIR / "subset_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 参数
N       = 521735
K_EVAL  = 20
K_SIM   = 50
W_TAG   = 0.5
W_DESC  = 0.3
W_CRE   = 0.2
DESC_THRESHOLD = 0.2

print(f"N={N}, K_EVAL={K_EVAL}, K_SIM={K_SIM}")
print(f"Unified weights: W_TAG={W_TAG}, W_DESC={W_DESC}, W_CRE={W_CRE}")
print(f"Output: {OUT_DIR}")

N=521735, K_EVAL=20, K_SIM=50
Unified weights: W_TAG=0.5, W_DESC=0.3, W_CRE=0.2
Output: /workspace/recsys-new/tmp/content/subset_eval


## 2. 加载 D_content 定义

In [2]:
d_content = pd.read_parquet(CONTENT_DIR / "d_content.parquet", engine="fastparquet")
d_content_set = set(d_content["doc_idx"].astype(int).values)
print(f"D_content: {len(d_content_set)} docs")
print(f"doc_idx range: [{min(d_content_set)}, {max(d_content_set)}]")

D_content: 1000 docs
doc_idx range: [551, 519823]


## 3. 加载银标准

In [3]:
# --- Tag 银标准 ---
tag_docs = pd.read_parquet(TMP_DIR / "relevance_tag_docs.parquet", engine="fastparquet")
tag_idf  = pd.read_parquet(TMP_DIR / "relevance_tag_idf.parquet", engine="fastparquet")

doc_tags = {}
for _, row in tag_docs.iterrows():
    idx = int(row["doc_idx"])
    tags = row["tags"]
    if isinstance(tags, list) and len(tags) > 0:
        doc_tags[idx] = tags
    elif isinstance(tags, str) and tags:
        doc_tags[idx] = [t.strip() for t in tags.split(",") if t.strip()]

idf_map = dict(zip(tag_idf["tag"], tag_idf["idf"]))
print(f"Tag silver standard: {len(doc_tags)} docs with tags, {len(idf_map)} unique tags")

# D_content 中有 tag 的文档数
d_content_with_tags = sum(1 for d in d_content_set if d in doc_tags)
print(f"D_content docs with tags: {d_content_with_tags}/{len(d_content_set)}")

Tag silver standard: 214585 docs with tags, 394 unique tags
D_content docs with tags: 1000/1000


In [4]:
# --- Desc 银标准（BM25） ---
print("Loading BM25 text similarity...")
S_bm25 = load_csr_from_manifest("S_textbm25_topk", N, TMP_DIR)
print(f"S_textbm25_topk: nnz={S_bm25.nnz:,}")

# D_content 中有 BM25 邻居的文档数
d_content_with_bm25 = sum(
    1 for d in d_content_set
    if S_bm25.indptr[d + 1] - S_bm25.indptr[d] > 0
)
print(f"D_content docs with BM25 neighbors: {d_content_with_bm25}/{len(d_content_set)}")

Loading BM25 text similarity...


S_textbm25_topk: nnz=19,859,800
D_content docs with BM25 neighbors: 1000/1000


In [5]:
# --- Creator 银标准 ---
beh_base = pd.read_parquet(TMP_DIR / "beh_base.parquet", engine="fastparquet")

creator_ids = np.zeros(N, dtype=np.int64)
for _, row in beh_base.iterrows():
    idx = int(row["doc_idx"])
    cid = int(row["CreatorUserId"])
    if 0 <= idx < N:
        creator_ids[idx] = cid

creator_counts = Counter(creator_ids[creator_ids > 0])
print(f"Creator silver standard: {(creator_ids > 0).sum()} docs with creator")

d_content_with_creator = sum(1 for d in d_content_set if creator_ids[d] > 0)
print(f"D_content docs with creator: {d_content_with_creator}/{len(d_content_set)}")

Creator silver standard: 521735 docs with creator
D_content docs with creator: 1000/1000


## 4. 子集评测函数

与 NB03 的 `evaluate_method` 类似，但硬性过滤 `doc_i in d_content_set`，
并记录每文档的详细分数用于后续分析。

In [6]:
def build_topk_for_method(prefix, k_eval, base_dir):
    """加载相似度边并返回每个文档的 top-K 邻居。"""
    man, parts, manifest_dir = load_manifest_flexible(prefix, base_dir, k=K_SIM)
    dfs = []
    for pf in parts:
        path = manifest_dir / pf
        if path.exists():
            dfs.append(pd.read_parquet(path, engine="fastparquet"))
    if not dfs:
        return {}, {}
    edges = pd.concat(dfs, ignore_index=True)
    nbr_idx = {}
    nbr_w = {}
    for row_i, group in edges.groupby("row"):
        row_i = int(row_i)
        top = group.nlargest(k_eval, "val")
        nbr_idx[row_i] = top["col"].values.astype(np.int64)
        nbr_w[row_i] = top["val"].values.astype(np.float32)
    return nbr_idx, nbr_w


def evaluate_method_subset(prefix, method_name, k_eval, base_dir, subset):
    """
    在指定子集上评测方法。
    只评测 doc_i in subset 的文档。
    返回 (results_dict, per_doc_records)。
    """
    nbr_idx, nbr_w = build_topk_for_method(prefix, k_eval, base_dir)
    if not nbr_idx:
        print(f"  WARNING: No neighbors loaded for {method_name}")
        return None, []

    results = {"method": method_name}
    per_doc = []  # 每文档的详细分数

    # ---- Tag 维度 ----
    tag_ndcgs, tag_maps, tag_mrrs, tag_precs, tag_recs = [], [], [], [], []
    tag_covered = 0

    for doc_i in subset:
        if doc_i not in nbr_idx:
            continue
        neighbors = nbr_idx[doc_i]
        if doc_i not in doc_tags:
            continue
        tags_i = set(doc_tags[doc_i])
        if not tags_i:
            continue
        tag_covered += 1

        gains = []
        binary = []
        for j in neighbors[:k_eval]:
            j = int(j)
            tags_j = set(doc_tags.get(j, []))
            inter = tags_i & tags_j
            union = tags_i | tags_j
            if union:
                idf_inter = sum(idf_map.get(t, 1.0) for t in inter)
                idf_union = sum(idf_map.get(t, 1.0) for t in union)
                gain = idf_inter / idf_union
            else:
                gain = 0.0
            gains.append(gain)
            binary.append(1.0 if len(inter) >= 1 else 0.0)

        gains = np.array(gains, dtype=np.float64)
        binary = np.array(binary, dtype=np.float64)
        ideal = np.sort(gains)[::-1]

        t_ndcg = ndcg_at_k(gains, ideal)
        t_map  = average_precision_at_k(binary)
        t_mrr  = mrr_at_k(binary)
        t_prec = precision_at_k(binary)
        t_rec  = float(binary.sum()) / max(k_eval, 1)

        tag_ndcgs.append(t_ndcg)
        tag_maps.append(t_map)
        tag_mrrs.append(t_mrr)
        tag_precs.append(t_prec)
        tag_recs.append(t_rec)

        # per-doc record
        per_doc.append({"doc_idx": doc_i, "method": method_name,
                        "tag_ndcg": t_ndcg, "tag_map": t_map})

    results["tag_ndcg"]  = np.mean(tag_ndcgs) if tag_ndcgs else 0.0
    results["tag_map"]   = np.mean(tag_maps)  if tag_maps  else 0.0
    results["tag_mrr"]   = np.mean(tag_mrrs)  if tag_mrrs  else 0.0
    results["tag_prec"]  = np.mean(tag_precs) if tag_precs else 0.0
    results["tag_rec"]   = np.mean(tag_recs)  if tag_recs  else 0.0
    results["tag_coverage"] = tag_covered / max(len(subset), 1)
    results["tag_n"]     = tag_covered

    # ---- Desc 维度 ----
    desc_ndcgs, desc_maps, desc_mrrs, desc_precs, desc_recs = [], [], [], [], []
    desc_covered = 0

    for doc_i in subset:
        if doc_i not in nbr_idx:
            continue
        neighbors = nbr_idx[doc_i]
        row_start = S_bm25.indptr[doc_i]
        row_end   = S_bm25.indptr[doc_i + 1]
        if row_end - row_start == 0:
            continue
        desc_covered += 1

        bm25_cols = S_bm25.indices[row_start:row_end]
        bm25_vals = S_bm25.data[row_start:row_end]
        bm25_lookup = dict(zip(bm25_cols.astype(int), bm25_vals.astype(float)))

        gains = []
        binary = []
        for j in neighbors[:k_eval]:
            j = int(j)
            sim = bm25_lookup.get(j, 0.0)
            gains.append(sim)
            binary.append(1.0 if sim > DESC_THRESHOLD else 0.0)

        gains = np.array(gains, dtype=np.float64)
        binary = np.array(binary, dtype=np.float64)
        ideal = np.sort(gains)[::-1]

        d_ndcg = ndcg_at_k(gains, ideal)
        d_map  = average_precision_at_k(binary)
        d_mrr  = mrr_at_k(binary)
        d_prec = precision_at_k(binary)
        d_rec  = float(binary.sum()) / max(k_eval, 1)

        desc_ndcgs.append(d_ndcg)
        desc_maps.append(d_map)
        desc_mrrs.append(d_mrr)
        desc_precs.append(d_prec)
        desc_recs.append(d_rec)

        # 更新 per-doc record
        for rec in per_doc:
            if rec["doc_idx"] == doc_i:
                rec["desc_ndcg"] = d_ndcg
                rec["desc_map"]  = d_map
                break

    results["desc_ndcg"]  = np.mean(desc_ndcgs) if desc_ndcgs else 0.0
    results["desc_map"]   = np.mean(desc_maps)  if desc_maps  else 0.0
    results["desc_mrr"]   = np.mean(desc_mrrs)  if desc_mrrs  else 0.0
    results["desc_prec"]  = np.mean(desc_precs) if desc_precs else 0.0
    results["desc_rec"]   = np.mean(desc_recs)  if desc_recs  else 0.0
    results["desc_coverage"] = desc_covered / max(len(subset), 1)
    results["desc_n"]     = desc_covered

    # ---- Creator 维度 ----
    cre_ndcgs, cre_maps, cre_mrrs, cre_precs, cre_recs = [], [], [], [], []
    cre_covered = 0

    for doc_i in subset:
        if doc_i not in nbr_idx:
            continue
        neighbors = nbr_idx[doc_i]
        cid_i = creator_ids[doc_i] if doc_i < N else 0
        if cid_i == 0:
            continue
        cre_covered += 1

        gains = []
        binary = []
        for j in neighbors[:k_eval]:
            j = int(j)
            cid_j = creator_ids[j] if j < N else 0
            match = 1.0 if (cid_j == cid_i and cid_j > 0) else 0.0
            gains.append(match)
            binary.append(match)

        gains = np.array(gains, dtype=np.float64)
        binary = np.array(binary, dtype=np.float64)
        ideal = np.sort(gains)[::-1]

        total_rel = max(creator_counts.get(cid_i, 1) - 1, 1)

        c_ndcg = ndcg_at_k(gains, ideal)
        c_map  = average_precision_at_k(binary)
        c_mrr  = mrr_at_k(binary)
        c_prec = precision_at_k(binary)
        c_rec  = recall_at_k(binary, total_rel)

        cre_ndcgs.append(c_ndcg)
        cre_maps.append(c_map)
        cre_mrrs.append(c_mrr)
        cre_precs.append(c_prec)
        cre_recs.append(c_rec)

        for rec in per_doc:
            if rec["doc_idx"] == doc_i:
                rec["cre_ndcg"] = c_ndcg
                rec["cre_map"]  = c_map
                break

    results["cre_ndcg"]  = np.mean(cre_ndcgs) if cre_ndcgs else 0.0
    results["cre_map"]   = np.mean(cre_maps)  if cre_maps  else 0.0
    results["cre_mrr"]   = np.mean(cre_mrrs)  if cre_mrrs  else 0.0
    results["cre_prec"]  = np.mean(cre_precs) if cre_precs else 0.0
    results["cre_rec"]   = np.mean(cre_recs)  if cre_recs  else 0.0
    results["cre_coverage"] = cre_covered / max(len(subset), 1)
    results["cre_n"]     = cre_covered

    # 统一 nDCG
    results["unified_ndcg"] = (
        W_TAG * results["tag_ndcg"]
        + W_DESC * results["desc_ndcg"]
        + W_CRE * results["cre_ndcg"]
    )

    print(f"  {method_name}: unified_nDCG={results['unified_ndcg']:.4f} "
          f"(tag={results['tag_ndcg']:.4f}, desc={results['desc_ndcg']:.4f}, "
          f"cre={results['cre_ndcg']:.4f}) "
          f"[tag_n={tag_covered}, desc_n={desc_covered}, cre_n={cre_covered}]")

    return results, per_doc

print("Subset evaluation function defined.")

Subset evaluation function defined.


## 5. 评测 8 种方法（限制在 D_content）

In [7]:
METHODS = {
    # --- 5 种融合方法 ---
    "Meta-only":       {"prefix": "S_fused3_symrow",      "dir": TMP_DIR},
    "Content-only":    {"prefix": "S_tabcontent_symrow",   "dir": CONTENT_DIR},
    "Naive-Fusion":    {"prefix": "S_naive4_symrow",       "dir": CONTENT_DIR},
    "Adaptive-Fusion": {"prefix": "S_fused4_symrow",       "dir": CONTENT_DIR},
    "Adaptive+Cons":   {"prefix": "S_fused4c_symrow",      "dir": CONTENT_DIR},
    # --- 3 种单视图 ---
    "Tag-only":        {"prefix": "S_tag_symrow",          "dir": TMP_DIR},
    "Text-only":       {"prefix": "S_text_symrow",         "dir": TMP_DIR},
    "Beh-only":        {"prefix": "S_beh_symrow",          "dir": TMP_DIR},
}

all_results = []
all_per_doc = []

for method_name, config in METHODS.items():
    print(f"\nEvaluating: {method_name}")
    try:
        res, pd_recs = evaluate_method_subset(
            config["prefix"], method_name, K_EVAL, config["dir"],
            subset=d_content_set,
        )
        if res is not None:
            all_results.append(res)
            all_per_doc.extend(pd_recs)
    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()

metrics_df = pd.DataFrame(all_results)
print(f"\nEvaluated {len(metrics_df)} methods on D_content subset.")


Evaluating: Meta-only


  Meta-only: unified_nDCG=0.4383 (tag=0.5763, desc=0.0077, cre=0.7391) [tag_n=1000, desc_n=1000, cre_n=1000]

Evaluating: Content-only


  Content-only: unified_nDCG=0.3889 (tag=0.7472, desc=0.0157, cre=0.0531) [tag_n=943, desc_n=943, cre_n=943]

Evaluating: Naive-Fusion


  Naive-Fusion: unified_nDCG=0.4749 (tag=0.6449, desc=0.0131, cre=0.7428) [tag_n=1000, desc_n=1000, cre_n=1000]

Evaluating: Adaptive-Fusion


  Adaptive-Fusion: unified_nDCG=0.4662 (tag=0.6257, desc=0.0113, cre=0.7496) [tag_n=1000, desc_n=1000, cre_n=1000]

Evaluating: Adaptive+Cons


  Adaptive+Cons: unified_nDCG=0.4662 (tag=0.6257, desc=0.0113, cre=0.7498) [tag_n=1000, desc_n=1000, cre_n=1000]

Evaluating: Tag-only


  Tag-only: unified_nDCG=0.1152 (tag=0.2302, desc=0.0000, cre=0.0004) [tag_n=1000, desc_n=1000, cre_n=1000]

Evaluating: Text-only


  Text-only: unified_nDCG=0.1127 (tag=0.2249, desc=0.0008, cre=0.0002) [tag_n=1000, desc_n=1000, cre_n=1000]

Evaluating: Beh-only


  Beh-only: unified_nDCG=0.4739 (tag=0.6416, desc=0.0097, cre=0.7506) [tag_n=1000, desc_n=1000, cre_n=1000]

Evaluated 8 methods on D_content subset.


## 6. 结果格式化打印

In [8]:
display_cols = [
    "method",
    "unified_ndcg",
    "tag_ndcg", "tag_map", "tag_prec", "tag_coverage",
    "desc_ndcg", "desc_map", "desc_prec", "desc_coverage",
    "cre_ndcg", "cre_map", "cre_prec", "cre_coverage",
]
avail_cols = [c for c in display_cols if c in metrics_df.columns]

print("=" * 120)
print("D_CONTENT SUBSET EVALUATION RESULTS")
print("=" * 120)

fmt = metrics_df[avail_cols].copy()
for c in avail_cols:
    if c != "method":
        fmt[c] = fmt[c].map(lambda x: f"{x:.4f}")
print(fmt.to_string(index=False))

D_CONTENT SUBSET EVALUATION RESULTS
         method unified_ndcg tag_ndcg tag_map tag_prec tag_coverage desc_ndcg desc_map desc_prec desc_coverage cre_ndcg cre_map cre_prec cre_coverage
      Meta-only       0.4383   0.5763  0.4874   0.2450       1.0000    0.0077   0.0048    0.0012        1.0000   0.7391  0.7339   0.2894       1.0000
   Content-only       0.3889   0.7472  0.7292   0.6414       0.9430    0.0157   0.0147    0.0015        0.9430   0.0531  0.0385   0.0078       0.9430
   Naive-Fusion       0.4749   0.6449  0.5790   0.4933       1.0000    0.0131   0.0099    0.0020        1.0000   0.7428  0.7368   0.2828       1.0000
Adaptive-Fusion       0.4662   0.6257  0.5382   0.2869       1.0000    0.0113   0.0092    0.0017        1.0000   0.7496  0.7430   0.2888       1.0000
  Adaptive+Cons       0.4662   0.6257  0.5386   0.2870       1.0000    0.0113   0.0092    0.0017        1.0000   0.7498  0.7435   0.2893       1.0000
       Tag-only       0.1152   0.2302  0.1221   0.0477       1.0

## 7. 保存结果

In [9]:
# 保存指标表
metrics_df.to_csv(OUT_DIR / "metrics_subset.csv", index=False)
print(f"Saved metrics_subset.csv: {len(metrics_df)} methods")

# 保存 per-doc 分数
if all_per_doc:
    per_doc_df = pd.DataFrame(all_per_doc)
    per_doc_df.to_parquet(OUT_DIR / "per_doc_scores.parquet", engine="fastparquet", index=False)
    print(f"Saved per_doc_scores.parquet: {len(per_doc_df)} records")
else:
    per_doc_df = pd.DataFrame()
    print("No per-doc records to save.")

Saved metrics_subset.csv: 8 methods
Saved per_doc_scores.parquet: 7943 records


## 8. 分析：内容视图优势 / 劣势文档

In [10]:
# 分析哪些文档内容视图显著优于/劣于元数据
if len(per_doc_df) > 0 and "tag_ndcg" in per_doc_df.columns:
    # 透视表：每文档 meta vs content nDCG
    meta_scores = per_doc_df[per_doc_df["method"] == "Meta-only"][["doc_idx", "tag_ndcg"]].rename(
        columns={"tag_ndcg": "meta_tag_ndcg"}
    )
    content_scores = per_doc_df[per_doc_df["method"] == "Content-only"][["doc_idx", "tag_ndcg"]].rename(
        columns={"tag_ndcg": "content_tag_ndcg"}
    )

    comparison = meta_scores.merge(content_scores, on="doc_idx", how="inner")
    comparison["delta"] = comparison["content_tag_ndcg"] - comparison["meta_tag_ndcg"]

    n_better  = (comparison["delta"] > 0.01).sum()
    n_worse   = (comparison["delta"] < -0.01).sum()
    n_similar = len(comparison) - n_better - n_worse

    print(f"\nPer-doc comparison (Content vs Meta, Tag nDCG):")
    print(f"  Content > Meta (+0.01): {n_better} docs ({100*n_better/len(comparison):.1f}%)")
    print(f"  Content ~ Meta:         {n_similar} docs ({100*n_similar/len(comparison):.1f}%)")
    print(f"  Content < Meta (-0.01): {n_worse} docs ({100*n_worse/len(comparison):.1f}%)")
    print(f"  Mean delta: {comparison['delta'].mean():.4f}")
    print(f"  Median delta: {comparison['delta'].median():.4f}")

    # Top-10 内容视图最优秀的文档
    print("\nTop-10 docs where Content >> Meta (Tag nDCG):")
    top_content = comparison.nlargest(10, "delta")
    for _, row in top_content.iterrows():
        print(f"  doc_idx={int(row['doc_idx']):>6d}: "
              f"meta={row['meta_tag_ndcg']:.4f}, content={row['content_tag_ndcg']:.4f}, "
              f"delta={row['delta']:+.4f}")

    # Top-10 元数据最优秀的文档
    print("\nTop-10 docs where Meta >> Content (Tag nDCG):")
    top_meta = comparison.nsmallest(10, "delta")
    for _, row in top_meta.iterrows():
        print(f"  doc_idx={int(row['doc_idx']):>6d}: "
              f"meta={row['meta_tag_ndcg']:.4f}, content={row['content_tag_ndcg']:.4f}, "
              f"delta={row['delta']:+.4f}")
else:
    comparison = pd.DataFrame()
    print("No per-doc comparison data available.")


Per-doc comparison (Content vs Meta, Tag nDCG):
  Content > Meta (+0.01): 643 docs (68.2%)
  Content ~ Meta:         28 docs (3.0%)
  Content < Meta (-0.01): 272 docs (28.8%)
  Mean delta: 0.1734
  Median delta: 0.1414

Top-10 docs where Content >> Meta (Tag nDCG):


  doc_idx=242847: meta=0.0000, content=0.9695, delta=+0.9695
  doc_idx=254966: meta=0.0000, content=0.9609, delta=+0.9609
  doc_idx=  2683: meta=0.0000, content=0.9547, delta=+0.9547
  doc_idx=  6247: meta=0.0000, content=0.9398, delta=+0.9398
  doc_idx=151783: meta=0.0000, content=0.9130, delta=+0.9130
  doc_idx=272269: meta=0.0000, content=0.9070, delta=+0.9070
  doc_idx=348835: meta=0.0000, content=0.9064, delta=+0.9064
  doc_idx=170275: meta=0.0000, content=0.9031, delta=+0.9031
  doc_idx= 57711: meta=0.0000, content=0.9009, delta=+0.9009
  doc_idx=  5021: meta=0.0000, content=0.9005, delta=+0.9005

Top-10 docs where Meta >> Content (Tag nDCG):
  doc_idx=  4257: meta=1.0000, content=0.5041, delta=-0.4959
  doc_idx= 97070: meta=1.0000, content=0.5266, delta=-0.4734
  doc_idx=475776: meta=0.9921, content=0.5567, delta=-0.4354
  doc_idx=340394: meta=0.9041, content=0.4691, delta=-0.4350
  doc_idx=102305: meta=1.0000, content=0.5698, delta=-0.4302
  doc_idx= 10840: meta=0.9330, content

In [11]:
# 融合方法分析：哪些文档融合优于单视图
if len(per_doc_df) > 0 and "tag_ndcg" in per_doc_df.columns:
    adaptive_scores = per_doc_df[per_doc_df["method"] == "Adaptive-Fusion"][["doc_idx", "tag_ndcg"]].rename(
        columns={"tag_ndcg": "adaptive_tag_ndcg"}
    )
    
    fusion_comp = meta_scores.merge(content_scores, on="doc_idx", how="inner").merge(
        adaptive_scores, on="doc_idx", how="inner"
    )
    fusion_comp["delta"] = fusion_comp["content_tag_ndcg"] - fusion_comp["meta_tag_ndcg"]
    fusion_comp["best_single"] = fusion_comp[["meta_tag_ndcg", "content_tag_ndcg"]].max(axis=1)
    fusion_comp["fusion_gain"] = fusion_comp["adaptive_tag_ndcg"] - fusion_comp["best_single"]
    
    n_fusion_better = (fusion_comp["fusion_gain"] > 0.01).sum()
    n_fusion_worse  = (fusion_comp["fusion_gain"] < -0.01).sum()
    
    print(f"\nFusion vs Best-Single-View (Tag nDCG):")
    print(f"  Fusion > Best-single (+0.01): {n_fusion_better} docs")
    print(f"  Fusion < Best-single (-0.01): {n_fusion_worse} docs")
    print(f"  Mean fusion gain: {fusion_comp['fusion_gain'].mean():.4f}")
else:
    fusion_comp = pd.DataFrame()
    print("No fusion comparison data available.")


Fusion vs Best-Single-View (Tag nDCG):
  Fusion > Best-single (+0.01): 200 docs
  Fusion < Best-single (-0.01): 663 docs
  Mean fusion gain: -0.1691


## 9. 一致性 vs 融合增益分析

In [12]:
# 加载一致性分数并与融合增益关联
consistency_path = CONTENT_DIR / "consistency_meta_content.parquet"
if consistency_path.exists() and len(fusion_comp) > 0:
    cons_df = pd.read_parquet(consistency_path, engine="fastparquet")
    
    # 合并一致性分数和融合增益
    analysis = fusion_comp.merge(
        cons_df[["doc_idx", "jaccard", "weighted_consistency"]],
        on="doc_idx", how="inner"
    )
    
    if len(analysis) > 0:
        # 按一致性分组
        analysis["cons_bin"] = pd.cut(
            analysis["jaccard"],
            bins=[0, 0.001, 0.005, 0.01, 1.0],
            labels=["0-0.001", "0.001-0.005", "0.005-0.01", ">0.01"]
        )
        
        print("\nFusion gain by consistency bin:")
        for grp_name, grp in analysis.groupby("cons_bin", observed=True):
            print(f"  Jaccard {grp_name}: n={len(grp)}, "
                  f"mean_fusion_gain={grp['fusion_gain'].mean():.4f}, "
                  f"mean_delta(cont-meta)={grp['delta'].mean():.4f}")
        
        # 相关系数
        corr_j = analysis["jaccard"].corr(analysis["fusion_gain"])
        corr_c = analysis["weighted_consistency"].corr(analysis["fusion_gain"])
        print(f"\nCorrelation(jaccard, fusion_gain): {corr_j:.4f}")
        print(f"Correlation(weighted_consistency, fusion_gain): {corr_c:.4f}")
    else:
        print("No analysis data after merge.")
        analysis = pd.DataFrame()
else:
    analysis = pd.DataFrame()
    if not consistency_path.exists():
        print("Consistency file not found.")
    else:
        print("No fusion comparison data for consistency analysis.")


Fusion gain by consistency bin:
  Jaccard 0.001-0.005: n=19, mean_fusion_gain=-0.1835, mean_delta(cont-meta)=0.2150
  Jaccard 0.005-0.01: n=114, mean_fusion_gain=-0.0729, mean_delta(cont-meta)=0.1627
  Jaccard >0.01: n=83, mean_fusion_gain=-0.0554, mean_delta(cont-meta)=0.1427

Correlation(jaccard, fusion_gain): 0.2021
Correlation(weighted_consistency, fusion_gain): 0.2133


## 10. 可视化

In [13]:
# --- 图 1: 子集方法对比柱状图 ---
if len(metrics_df) > 0:
    fig, ax = plt.subplots(figsize=(16, 7))

    methods = metrics_df["method"].tolist()
    x = np.arange(len(methods))
    width = 0.18

    dims = [
        ("tag_ndcg",     "Tag nDCG",     "#4C72B0"),
        ("desc_ndcg",    "Desc nDCG",    "#55A868"),
        ("cre_ndcg",     "Creator nDCG", "#C44E52"),
        ("unified_ndcg", "Unified nDCG", "#8172B2"),
    ]

    for i, (col, label, color) in enumerate(dims):
        if col in metrics_df.columns:
            vals = metrics_df[col].values
            bars = ax.bar(x + i * width, vals, width, label=label, color=color, alpha=0.85)
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                        f"{val:.3f}", ha="center", va="bottom", fontsize=6, rotation=45)

    ax.set_xlabel("Method")
    ax.set_ylabel("nDCG@20")
    ax.set_title("D_content Subset Evaluation: Method Comparison (1000 docs)")
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(methods, rotation=25, ha="right")
    ax.legend(loc="upper right")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fig.savefig(OUT_DIR / "fig_subset_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig_subset_comparison.png")
else:
    print("No metrics data for visualization.")

Saved fig_subset_comparison.png


In [14]:
# --- 图 2: Per-doc scatter（Meta vs Content Tag nDCG） ---
if len(comparison) > 0:
    fig, ax = plt.subplots(figsize=(8, 8))

    ax.scatter(
        comparison["meta_tag_ndcg"],
        comparison["content_tag_ndcg"],
        alpha=0.4, s=15, c="steelblue", edgecolors="none",
    )
    # 对角线
    lims = [0, 1]
    ax.plot(lims, lims, "k--", alpha=0.4, linewidth=1)

    ax.set_xlabel("Meta-only Tag nDCG")
    ax.set_ylabel("Content-only Tag nDCG")
    ax.set_title(f"Per-doc: Meta vs Content (n={len(comparison)})")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)

    # 标注比例
    n_above = (comparison["content_tag_ndcg"] > comparison["meta_tag_ndcg"] + 0.01).sum()
    n_below = (comparison["content_tag_ndcg"] < comparison["meta_tag_ndcg"] - 0.01).sum()
    ax.text(0.05, 0.92, f"Content > Meta: {n_above}",
            transform=ax.transAxes, fontsize=10, color="green")
    ax.text(0.05, 0.87, f"Meta > Content: {n_below}",
            transform=ax.transAxes, fontsize=10, color="red")

    plt.tight_layout()
    fig.savefig(OUT_DIR / "fig_meta_vs_content_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig_meta_vs_content_scatter.png")
else:
    print("No comparison data for scatter plot.")

Saved fig_meta_vs_content_scatter.png


In [15]:
# --- 图 3: 一致性 vs 融合增益 ---
if len(analysis) > 0 and "jaccard" in analysis.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # 左图：Jaccard vs fusion gain scatter
    ax = axes[0]
    ax.scatter(
        analysis["jaccard"], analysis["fusion_gain"],
        alpha=0.4, s=15, c="darkorange", edgecolors="none",
    )
    ax.axhline(0, color="k", linestyle="--", alpha=0.4)
    ax.set_xlabel("Jaccard(meta, content) neighbors")
    ax.set_ylabel("Fusion gain over best single view (Tag nDCG)")
    ax.set_title("Consistency vs Fusion Gain")
    ax.grid(alpha=0.3)

    # 右图：按一致性分组的融合增益箱线图
    ax = axes[1]
    groups = analysis.groupby("cons_bin", observed=True)["fusion_gain"]
    labels_list = []
    data_list = []
    for name, grp in groups:
        labels_list.append(f"{name}\n(n={len(grp)})")
        data_list.append(grp.values)

    if data_list:
        bp = ax.boxplot(data_list, labels=labels_list, patch_artist=True)
        for patch in bp["boxes"]:
            patch.set_facecolor("lightsalmon")
        ax.axhline(0, color="k", linestyle="--", alpha=0.4)
        ax.set_xlabel("Jaccard consistency bin")
        ax.set_ylabel("Fusion gain (Tag nDCG)")
        ax.set_title("Fusion Gain by Consistency Level")
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fig.savefig(OUT_DIR / "fig_consistency_vs_gain.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig_consistency_vs_gain.png")
else:
    print("No consistency-gain analysis data for visualization.")

Saved fig_consistency_vs_gain.png


/tmp/ipykernel_157506/1789522369.py:27: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_list, labels=labels_list, patch_artist=True)


## 11. 全局 vs 子集对比

In [16]:
# 对比全局评测和子集评测结果
global_metrics_path = CONTENT_DIR / "metrics_content_experiments.csv"
if global_metrics_path.exists():
    global_df = pd.read_csv(global_metrics_path)

    print("\n" + "=" * 100)
    print("GLOBAL vs SUBSET COMPARISON")
    print("=" * 100)

    for method_name in ["Meta-only", "Content-only", "Naive-Fusion", "Adaptive-Fusion", "Adaptive+Cons"]:
        g_row = global_df[global_df["method"] == method_name]
        s_row = metrics_df[metrics_df["method"] == method_name]
        if len(g_row) == 0 or len(s_row) == 0:
            continue

        g_unified = g_row["unified_ndcg"].values[0]
        s_unified = s_row["unified_ndcg"].values[0]
        g_tag = g_row["tag_ndcg"].values[0]
        s_tag = s_row["tag_ndcg"].values[0]

        print(f"\n{method_name}:")
        print(f"  Global:  unified={g_unified:.4f}, tag={g_tag:.4f}")
        print(f"  Subset:  unified={s_unified:.4f}, tag={s_tag:.4f}")
        print(f"  Delta:   unified={s_unified - g_unified:+.4f}, tag={s_tag - g_tag:+.4f}")
else:
    print("Global metrics file not found. Skipping comparison.")


GLOBAL vs SUBSET COMPARISON

Meta-only:
  Global:  unified=0.3064, tag=0.3157
  Subset:  unified=0.4383, tag=0.5763
  Delta:   unified=+0.1319, tag=+0.2606

Content-only:
  Global:  unified=0.3889, tag=0.7472
  Subset:  unified=0.3889, tag=0.7472
  Delta:   unified=+0.0000, tag=+0.0000

Naive-Fusion:
  Global:  unified=0.3049, tag=0.3144
  Subset:  unified=0.4749, tag=0.6449
  Delta:   unified=+0.1700, tag=+0.3305

Adaptive-Fusion:
  Global:  unified=0.3047, tag=0.3141
  Subset:  unified=0.4662, tag=0.6257
  Delta:   unified=+0.1615, tag=+0.3117

Adaptive+Cons:
  Global:  unified=0.3047, tag=0.3141
  Subset:  unified=0.4662, tag=0.6257
  Delta:   unified=+0.1615, tag=+0.3116


## 12. 总结

In [17]:
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print(f"\nSubset size: {len(d_content_set)} docs")
print(f"Evaluation K: {K_EVAL}")
print(f"Methods evaluated: {len(metrics_df)}")

if len(metrics_df) > 0:
    best_unified = metrics_df.loc[metrics_df["unified_ndcg"].idxmax()]
    best_tag     = metrics_df.loc[metrics_df["tag_ndcg"].idxmax()]

    print(f"\nBest unified nDCG: {best_unified['method']} = {best_unified['unified_ndcg']:.4f}")
    print(f"Best tag nDCG:     {best_tag['method']} = {best_tag['tag_ndcg']:.4f}")

    # 融合 vs 单视图
    meta_ndcg = metrics_df[metrics_df["method"] == "Meta-only"]["unified_ndcg"].values
    content_ndcg = metrics_df[metrics_df["method"] == "Content-only"]["unified_ndcg"].values
    adaptive_ndcg = metrics_df[metrics_df["method"] == "Adaptive-Fusion"]["unified_ndcg"].values

    if len(meta_ndcg) > 0 and len(adaptive_ndcg) > 0:
        print(f"\nAdaptive-Fusion vs Meta-only (unified): {adaptive_ndcg[0] - meta_ndcg[0]:+.4f}")
    if len(content_ndcg) > 0 and len(adaptive_ndcg) > 0:
        print(f"Adaptive-Fusion vs Content-only (unified): {adaptive_ndcg[0] - content_ndcg[0]:+.4f}")

print(f"\nOutput files:")
for f in sorted(OUT_DIR.glob("*")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

print("\nDone.")


SUMMARY

Subset size: 1000 docs
Evaluation K: 20
Methods evaluated: 8

Best unified nDCG: Naive-Fusion = 0.4749
Best tag nDCG:     Content-only = 0.7472

Adaptive-Fusion vs Meta-only (unified): +0.0279
Adaptive-Fusion vs Content-only (unified): +0.0773

Output files:
  fig_consistency_vs_gain.png (120.8 KB)
  fig_meta_vs_content_scatter.png (181.3 KB)
  fig_subset_comparison.png (114.1 KB)
  metrics_subset.csv (2.7 KB)
  per_doc_scores.parquet (119.0 KB)

Done.
